# Demo 2 · частина 2: де насправді ламається детектор

До слайдів **«Який найменший об'єкт бачить детектор»** і **«Як збільшити дальність детекції»**. Продовження [частини 1](lesson_01_demo2_part1_detection_range.ipynb): там дальність рахувалась із правила «~16 px на ціль», тут перевіряємо це правило справжнім YOLO.

- **Блок 1. Поріг детекції** — з якого розміру в пікселях модель перестає бачити ціль. Слайд обіцяє: детекція стабільна від ~16 px, класифікація — від ~32 px.
- **Блок 2. Як повернути дальність** — tiled inference і теле-об'єктив на одному кадрі, разом із ціною кожного.

Обидва блоки одразу відкриваються з фото з `artifacts/images/`. Своє можна обрати кнопкою **«Обрати фото»**. Для блоку 2 потрібен кадр із дрібними об'єктами (вулиця згори, парковка, аерофото), інакше tiled не дасть виграшу.

### Що потрібно

- `ultralytics` і `torch` — є в базовому середовищі: `scripts/setup.ps1` (Windows) або `scripts/setup.sh` (Ubuntu, macOS);
- ваги `yolo11m.pt` в `artifacts/models/` — поклади заздалегідь, бо на занятті може не бути інтернету;
- `ipyevents` — щоб обирати ціль мишею; без нього лишаються слайдери.

In [11]:
from ultralytics import YOLO

from panels import model_path

CANVAS = 640  # вхід моделі; тримаємо сталим, щоб єдиною змінною був розмір цілі
MODEL = YOLO(model_path("yolo11m.pt"))  # ваги беремо з artifacts/models


def detect(image, conf: float = 0.05, imgsz: int = CANVAS):
    """Детекції як список (клас, впевненість, bbox xyxy) у координатах переданого кадру."""
    result = MODEL.predict(image, imgsz=imgsz, conf=conf, verbose=False)[0]
    return [(result.names[int(box.cls)], float(box.conf), tuple(float(v) for v in box.xyxy[0]))
            for box in result.boxes]

## Блок 1: поріг детекції

**Як це працює.** Обираєш на фото ціль, а панель «віддаляє» її: масштабує сцену так, щоб ціль займала 64, 48, … 8 px, і на кожному розмірі запускає детектор. Вхід моделі весь час той самий (640 px), тож змінюється лише розмір цілі. Крива впевненості показує, на якому розмірі модель втрачає ціль, а формула з частини 1 переводить цей поріг у метри.

### Пікселі → метри

Та сама формула, що в частині 1: на якій дистанції ціль займе `min_px` пікселів на вході моделі за обраної оптики.

In [13]:
UM_PER_MM = 1000  # p задано в мкм, а f — у мм


def detection_range(target_m: float, focal_mm: float, pixel_um: float,
                    resize_k: float, min_px: float) -> float:
    """Дистанція, на якій ціль ще займає min_px пікселів на вході моделі."""
    return UM_PER_MM * focal_mm * target_m / (pixel_um * resize_k * min_px)

### Імітація дистанції

- `scene_at_size` — кадр 640×640, у якому ціль займає рівно N пікселів. Усе, що виходить за межі фото, заповнюється сірим.
- `matched_confidence` — впевненість детектора саме на нашій цілі, а не на іншому об'єкті в кадрі.

In [15]:
import cv2
import numpy as np

PAD_GRAY = 114  # той самий нейтральний сірий, яким YOLO робить letterbox


def scene_at_size(image: np.ndarray, object_px: float, target_px: float,
                  center_xy: tuple[float, float], canvas: int = CANVAS) -> np.ndarray:
    """Кадр canvas×canvas, у якому ціль займає рівно target_px пікселів.

    Масштабуємо всю сцену й вирізаємо вікно навколо цілі. Те, що виходить за межі фото,
    заповнюємо сірим: контексту за кадром немає, і домальовувати його було б нечесно.
    """
    scale = target_px / object_px
    x0 = int(round(center_xy[0] * scale)) - canvas // 2  # вікно в координатах масштабованого фото
    y0 = int(round(center_xy[1] * scale)) - canvas // 2

    if scale > 1:
        # збільшувати все фото заради вікна canvas×canvas — це гігабайти пам'яті на великому кадрі,
        # тож рахуємо одразу вікно: те саме перетворення, що resize + кроп, без проміжного фото
        shift = 0.5 * scale - 0.5  # центри пікселів, як у cv2.resize
        warp = np.float32([[scale, 0, shift - x0], [0, scale, shift - y0]])
        return cv2.warpAffine(image, warp, (canvas, canvas), flags=cv2.INTER_CUBIC,
                              borderMode=cv2.BORDER_CONSTANT, borderValue=(PAD_GRAY,) * 3)

    scaled = cv2.resize(image, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    frame = np.full((canvas, canvas, 3), PAD_GRAY, dtype=np.uint8)
    sx0, sy0 = max(0, x0), max(0, y0)
    sx1, sy1 = min(scaled.shape[1], x0 + canvas), min(scaled.shape[0], y0 + canvas)
    if sx1 > sx0 and sy1 > sy0:
        frame[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0] = scaled[sy0:sy1, sx0:sx1]
    return frame


def matched_confidence(detections, target_cls: str | None, expected_px: float,
                       canvas: int = CANVAS) -> float:
    """Найвища впевненість серед детекцій, чий центр біля центру кадру.

    target_cls=None — будь-який клас: для вручну обведеної цілі ми не знаємо, чим її вважає
    модель (дрон вона цілком може назвати птахом або літаком).
    """
    center = canvas / 2
    tolerance = max(expected_px, 12)
    return max((conf for cls, conf, (x1, y1, x2, y2) in detections
                if (target_cls is None or cls == target_cls)
                and np.hypot((x1 + x2) / 2 - center, (y1 + y2) / 2 - center) <= tolerance),
               default=0.0)

### Панель

1. Фото вже завантажене. Сірі рамки — об'єкти, які знайшов детектор, зелена — поточна ціль.
2. Клацни по рамці, щоб обрати іншу ціль, або протягни мишею нову — так можна взяти й об'єкт поза COCO (дрон модель може назвати птахом).
3. Натисни **«Виміряти поріг»**.
4. У розділі **«Переведення пікселів у метри»** введи оптику своєї камери — панель порахує, на якій дистанції модель втратить ціль.

In [29]:
from panels import launch_threshold

launch_threshold(detect, scene_at_size, matched_confidence, detection_range, CANVAS)

## Блок 2: як повернути дальність

До слайда **«Як збільшити дальність детекції»**. Той самий кадр у трьох режимах:

| Режим | Як працює | Ціна |
|---|---|---|
| **Базовий resize** | усе фото стискається до входу моделі, дрібні цілі зникають | 1 прохід |
| **Tiled (SAHI)** | фото ріжеться на тайли, і кожен іде в модель без стиснення, `k = 1` | прохід на тайл |
| **Теле** | широкий кадр + вузький центр, рамки обох зливаються | 2 проходи |

Порядок той самий, що радить слайд: спершу оптика й пайплайн, а вже потім важча модель і більший вхід.

Тайли перекриваються, щоб об'єкт на стику не розрізало навпіл, тож той самий об'єкт може прийти з двох тайлів — дублікати прибирає NMS. З теле так само: центр кадру бачать обидва проходи.

In [21]:
def axis_starts(length: int, tile: int, step: int) -> list[int]:
    """Початки тайлів уздовж однієї осі; останній притискаємо до краю, щоб не вийти за кадр."""
    starts = list(range(0, max(1, length - tile + 1), step))
    if starts[-1] + tile < length:
        starts.append(length - tile)
    return starts


def tile_origins(width: int, height: int, tile: int, overlap: float) -> list[tuple[int, int]]:
    """Ліві верхні кути тайлів."""
    step = max(1, int(tile * (1 - overlap)))
    return [(max(0, x), max(0, y))
            for y in axis_starts(height, tile, step) for x in axis_starts(width, tile, step)]


def iou(a: tuple, b: tuple) -> float:
    """Перетин до об'єднання для двох bbox xyxy."""
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    if inter == 0:
        return 0.0
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (area_a + area_b - inter)


def is_new(detection: tuple, known: list, iou_threshold: float = 0.5) -> bool:
    """Чи це справді нова знахідка, а не той самий об'єкт, який уже є в known."""
    cls, _, box = detection
    return all(known_cls != cls or iou(box, known_box) < iou_threshold
               for known_cls, _, known_box in known)


def merge_detections(detections: list, iou_threshold: float = 0.5) -> list:
    """NMS по класах: сусідні тайли перекриваються, тож один об'єкт приходить двічі."""
    kept = []
    for detection in sorted(detections, key=lambda d: -d[1]):  # спочатку найвпевненіші
        if is_new(detection, kept, iou_threshold):
            kept.append(detection)
    return kept

### Три режими

Кожен режим повертає `(детекції, кількість проходів)`, тож видно не лише виграш, а й ціну. Теле повертає ще й третє значення — об'єкти, які знайшов лише він; панель виділить їх яскравою рамкою.

In [24]:
def shift(detections: list, dx: int, dy: int) -> list:
    """Бокси з координат кропу — у координати всього фото."""
    return [(cls, conf, (x1 + dx, y1 + dy, x2 + dx, y2 + dy))
            for cls, conf, (x1, y1, x2, y2) in detections]


def run_whole(image, detect, imgsz: int = CANVAS) -> tuple[list, int]:
    """Базовий режим: усе фото за один прохід, модель сама стисне його до imgsz."""
    return detect(image, imgsz=imgsz), 1


def run_tiled(image, detect, tile: int = CANVAS, overlap: float = 0.2) -> tuple[list, int]:
    """Tiled inference: кожен тайл іде в модель у рідній роздільності, тобто без стиснення."""
    img_h, img_w = image.shape[:2]
    if img_w <= tile and img_h <= tile:
        return detect(image, imgsz=tile), 1  # кадр і так менший за вхід — різати нічого
    tile = min(tile, img_w, img_h)

    origins = tile_origins(img_w, img_h, tile, overlap)
    found = []
    for x0, y0 in origins:
        found += shift(detect(image[y0:y0 + tile, x0:x0 + tile], imgsz=tile), x0, y0)
    return merge_detections(found), len(origins)


def run_tele(image, detect, zoom: int, imgsz: int = CANVAS) -> tuple[list, int, list]:
    """Широкий кадр + теле-кроп центру: два проходи, рамки злиті в один набір.

    Третім значенням віддаємо те, що знайшлося лише в теле, — виграш від довшої фокусної в об'єктах.
    """
    wide, _ = run_whole(image, detect, imgsz)

    img_h, img_w = image.shape[:2]
    crop_w, crop_h = int(img_w / zoom), int(img_h / zoom)
    x0, y0 = (img_w - crop_w) // 2, (img_h - crop_h) // 2
    tele = shift(detect(image[y0:y0 + crop_h, x0:x0 + crop_w], imgsz=imgsz), x0, y0)

    gained = [detection for detection in tele if is_new(detection, wide)]
    return merge_detections(wide + tele), 2, gained

### Порівняння

Натисни **«Порівняти режими»**. На великому фото tiled рахує помітно довше — це і є його ціна, її видно у стовпчиках «Проходів детектора» і «Час, с».

In [27]:
from panels import launch_levers

launch_levers(detect, run_whole, run_tiled, run_tele)